# SAC Irrigation Training — v2.8.0 (Kaggle)

## Key changes from v2.7
| # | Change | Effect |
|---|--------|--------|
| 1 | `x1_overshoot_norm` added as 9th per-agent feature | Direct gradient signal from r6 reward |
| 2 | Episode-length curriculum (60d → 93d at step 50k) | Reduces critic-target variance during initial training |
| 3 | OBS_DIM 1097 → 1227 | per-agent block 8 → 9 features |
| 4 | TOTAL_TIMESTEPS 500k → 250k | v2.7 peaked at step 200k on both seeds |

## Diagnostic checks before full training
Cell 3 runs all tests, a 25k pilot, and the obs-dim sanity check. Expected at step 25k:
- `critic_loss < 100` (curriculum should suppress the v2.7 explosion)
- `ep_len_mean == 60` (curriculum still in warmup at step 25k)
- `ent_coef` constant at 0.05

After step 50k the curriculum switches to 93-day episodes; `ep_len_mean` should reach 93 by step 60k.

## Seed plan (paired-samples design)
**Use the SAME seeds as the v2.7 baseline** (start with 0 and 1, then extend to 2, 3, 4). This is a paired-samples design: directly compare v2.8_seed_i vs v2.7_seed_i to control for initialization randomness, then average across seeds for the overall protocol effect. The v2.7 baseline results in `results/runs/sac_perfect_det_*_seed{0,1}.json` are NOT overwritten by v2.8 training (v2.8 writes to `results/rl/sac_v28_seed{N}/`).

**Change `SEED` to 0, 1, 2, 3, 4 in Cell 4 and run one Kaggle notebook per seed.**

**Submit as Save Version → Save & Run All** to avoid browser-disconnect zombie sessions.

In [ ]:
# ── Cell 1: Install & clone ──────────────────────────────────────────────────
import subprocess, sys, os, torch

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

run('pip install stable-baselines3==2.6.0 gymnasium wandb pytest --quiet')

if os.path.exists('/kaggle/working/thesis'):
    run('cd /kaggle/working/thesis && git pull')
else:
    run('git clone https://github.com/taratorbati/thesis.git /kaggle/working/thesis')

os.chdir('/kaggle/working/thesis')
sys.path.insert(0, '/kaggle/working/thesis')

import numpy as np, gymnasium, stable_baselines3 as sb3
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA:              {torch.cuda.is_available()}')
torch.set_num_threads(4)
print('Setup complete.')

In [ ]:
# ── Cell 2: WandB secret ────────────────────────────────────────────────────
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('✓  WandB API key loaded.')
except Exception as e:
    print(f'⚠  Could not load WandB key ({e}); training will continue without WandB.')

In [ ]:
# ── Cell 3: Tests + 25k pilot (mandatory pre-flight) ────────────────────────
#
# Abort if any of the following fail:
#   - any test fails
#   - obs_dim != 1227
#   - critic_loss > 100 at step 25k (curriculum should suppress explosion)
#   - ep_len_mean != 60 at step 25k (curriculum should be active)

import subprocess, sys

print('Running smoke + VDN unit tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest',
     'tests/test_rl_smoke.py', 'tests/test_factorized_critic.py',
     '-v', '--tb=short'],
    capture_output=False
)
if r.returncode != 0:
    raise RuntimeError('TESTS FAILED — do not proceed')
print()

from src.rl.gym_env import IrrigationEnv, OBS_DIM
env_check = IrrigationEnv(randomize=False)
obs_check, _ = env_check.reset()
assert obs_check.shape[0] == 1227, f'Wrong obs_dim: {obs_check.shape[0]} (expected 1227)'
print(f'obs_dim: {obs_check.shape[0]} ✓')
print()

print('Running 25k pilot...')
from src.rl.train import train_sac
train_sac(
    seed=0,
    output_dir='/kaggle/working/thesis/results/rl_pilot',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=25_000,
    curriculum_warmup_steps=50_000,   # curriculum will be active throughout the pilot
    curriculum_short_len=60,
)
print()
print('✓  Pilot complete. Check WandB:')
print('   critic_loss < 100         →  proceed')
print('   ep_len_mean == 60         →  curriculum is active')
print('   Either condition fails    →  STOP and investigate')

In [ ]:
# ── Cell 4: Full 250k training ───────────────────────────────────────────────
# Only run after Cell 3 pilot passed.
# Change SEED to 0, 1, 2, 3, or 4 for each parallel session.
# Pair v2.8 seeds with v2.7 baseline seeds (paired-samples design).

SEED = 0   # ← CHANGE THIS: use 0, 1, 2, 3, 4 (paired with v2.7 baseline)

from src.rl.train import train_sac

model = train_sac(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    curriculum_warmup_steps=50_000,
    curriculum_short_len=60,
)
print(f'Training complete for seed {SEED}.')

In [ ]:
# ── Cell 5: Copy results to output (replay buffer excluded) ──────────────────
import shutil, os

src = f'/kaggle/working/thesis/results/rl/sac_v28_seed{SEED}'
dst = f'/kaggle/working/results_v28_seed{SEED}'

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))

files = [f for root, _, files in os.walk(dst) for f in files]
print(f'Results copied to {dst} ({len(files)} files).')
for root, _, flist in os.walk(dst):
    for f in flist:
        path = os.path.join(root, f)
        rel  = os.path.relpath(path, dst)
        print(f'  {rel}  ({os.path.getsize(path)/1e6:.2f} MB)')

In [ ]:
# ── Cell 6: Zip & download ───────────────────────────────────────────────────
import shutil
archive = f'/kaggle/working/sac_v28_seed{SEED}'
shutil.make_archive(archive, 'zip', f'/kaggle/working/results_v28_seed{SEED}')
print(f'Archive ready: {archive}.zip')
print('Download via Kaggle → Output → Files panel.')

In [ ]:
# ── Cell 7: Resume from checkpoint (if session was killed) ───────────────────
# Uncomment, fill in, and run if session was interrupted.

# SEED = 2
# CHECKPOINT_STEPS = 100_000
# CHECKPOINT_ZIP = f'/kaggle/input/sac-v28-checkpoint/sac_v28_seed{SEED}_{CHECKPOINT_STEPS}_steps.zip'
# BUFFER_PKL     = f'/kaggle/input/sac-v28-checkpoint/replay_buffer_latest.pkl'
#
# from stable_baselines3 import SAC
# from src.rl.train import _make_lr_schedule, LR_START, LR_END
# from stable_baselines3.common.vec_env import DummyVecEnv
# from src.rl.gym_env import IrrigationEnv
#
# env = DummyVecEnv([lambda: IrrigationEnv(randomize=True, curriculum_warmup_steps=0)])
# model = SAC.load(CHECKPOINT_ZIP, env=env)
# model.load_replay_buffer(BUFFER_PKL)
# model.lr_schedule = _make_lr_schedule(LR_START, LR_END)
# remaining = 250_000 - CHECKPOINT_STEPS
# model.learn(total_timesteps=remaining, reset_num_timesteps=False, progress_bar=True)
# model.save(f'/kaggle/working/thesis/results/rl/sac_v28_seed{SEED}/sac_v28_seed{SEED}_final')
# print('Resume complete.')